# Chapter 38 — Representing Images and Geometry

Companion tutorial for **[Foundations of Computer Vision](https://visionbook.mit.edu/homogeneous_coordinates.html)** by Torralba, Isola, and Freeman (MIT Press, 2024).

This is the foundational chapter for everything in Part XI (geometry). Three ideas drive it:

1. **Homogeneous coordinates** let you write translation, rotation, scaling, shearing, and projective warp as a single matrix multiplication.
2. **Lines and points are dual** under the cross product in homogeneous form — joining two points and intersecting two lines are the same operation.
3. **Implicit image representations** (SIREN-style MLPs that map $(x,y)\to \text{RGB}$) give you sub-pixel image access without any interpolation kernel — geometric transformations become inverse-coordinate evaluations.

We reproduce all 12 figures. The most interesting bits live at the ends: figure 38.8 reframes warping as a 1D convolution on coordinates, and figure 38.12 swaps the entire pixel grid for a neural network.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, FancyArrow

torch.set_default_dtype(torch.float32)
torch.manual_seed(0)
np.random.seed(0)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": False,
})

## 38.2 — Homogeneous and heterogeneous coordinates

Heterogeneous coordinates write a 2D point as $(x, y)$. Homogeneous coordinates write the same point as $(x, y, 1)$ — and crucially, also as $(\lambda x, \lambda y, \lambda)$ for any $\lambda \ne 0$. All points on the ray through the origin and $(x, y, 1)$ represent the same 2D point. Conversion back is divide-by-$w$:

$$\begin{bmatrix}x\\y\\w\end{bmatrix} \;\longrightarrow\; \left(\frac{x}{w}, \frac{y}{w}\right).$$

The payoff: translation, which is an *addition* in heterogeneous coords, becomes a *multiplication* in homogeneous coords. That uniformity lets you compose any sequence of geometric transformations as a single matrix product.

In [ ]:
def to_homogeneous(p):
    """Append a 1 to the last axis.  (..., D) -> (..., D+1)."""
    return torch.cat([p, torch.ones_like(p[..., :1])], dim=-1)


def from_homogeneous(p):
    """Divide by last coord. (..., D+1) -> (..., D)."""
    return p[..., :-1] / p[..., -1:]

In [ ]:
# Figure 38.1 — the ray (lambda*x, lambda*y, lambda) all maps to one 2D point.
fig = plt.figure(figsize=(7, 4))
ax = fig.add_subplot(111, projection="3d")

x, y = 1.5, 0.8
lambdas = np.linspace(0.3, 1.8, 14)
ray = np.stack([lambdas * x, lambdas * y, lambdas], axis=1)
ax.plot(ray[:, 0], ray[:, 1], ray[:, 2], "k--", lw=0.7)
ax.scatter(ray[:, 0], ray[:, 1], ray[:, 2], color="crimson", s=20)
ax.scatter([x], [y], [1], color="navy", s=80)
ax.text(x + 0.1, y, 1.05, r"$(x,y,1)$", color="navy")

# w=1 plane
xx, yy = np.meshgrid(np.linspace(0, 2.5, 5), np.linspace(-0.2, 1.5, 5))
ax.plot_surface(xx, yy, np.ones_like(xx), alpha=0.1, color="steelblue")
ax.text(2.4, 0.0, 1.05, r"$w=1$ plane", color="steelblue")

ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("w")
ax.set_title("Figure 38.1 — all points on the ray represent the same (x, y)")
ax.view_init(elev=22, azim=-58)
plt.tight_layout()
plt.show()

## 38.3 — 2D image transformations

Every transformation that follows is a $3\times 3$ matrix acting on homogeneous points. We'll use one reference shape — a stylised "F" — and apply each transformation to it.

In [ ]:
# Reference shape: a stylised "F" (asymmetric so we can see flips/shears).
REF = torch.tensor([
    [0.0, 0.0], [0.0, 2.0], [1.2, 2.0], [1.2, 1.6], [0.4, 1.6],
    [0.4, 1.1], [1.0, 1.1], [1.0, 0.7], [0.4, 0.7], [0.4, 0.0], [0.0, 0.0],
])


def apply_T(M, pts):
    """Apply 3x3 homogeneous matrix to (N, 2) points."""
    return from_homogeneous(to_homogeneous(pts) @ M.T)


def T_translate(tx, ty):
    return torch.tensor([[1.0, 0.0, tx], [0.0, 1.0, ty], [0.0, 0.0, 1.0]])


def T_scale(sx, sy):
    return torch.tensor([[sx, 0.0, 0.0], [0.0, sy, 0.0], [0.0, 0.0, 1.0]])


def T_rotate(theta):
    c, s = float(np.cos(theta)), float(np.sin(theta))
    return torch.tensor([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])


def T_shear(shx, shy):
    return torch.tensor([[1.0, shx, 0.0], [shy, 1.0, 0.0], [0.0, 0.0, 1.0]])

In [ ]:
# Figure 38.2 — summary of 2D transformations applied to the reference shape.
fig, axes = plt.subplots(1, 5, figsize=(13, 3))
transforms = [
    ("reference", torch.eye(3)),
    ("translation", T_translate(0.8, 0.4)),
    ("scaling", T_scale(0.6, 1.4)),
    ("rotation", T_rotate(np.pi / 6)),
    ("shearing", T_shear(0.35, 0.0)),
]
for ax, (name, M) in zip(axes, transforms):
    pts = apply_T(M, REF).numpy()
    ax.add_patch(Polygon(REF.numpy(), closed=True, facecolor="lightgray", edgecolor="gray", lw=0.8))
    ax.add_patch(Polygon(pts, closed=True, facecolor=(0.85, 0.2, 0.2, 0.35), edgecolor="crimson", lw=1.5))
    ax.set_xlim(-0.5, 2.7); ax.set_ylim(-0.5, 2.7); ax.set_aspect("equal")
    ax.set_title(name)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Figure 38.2 — five 2D geometric transformations")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 38.3 — single translation and composition of two translations.
fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))

T1 = T_translate(0.7, 0.3)
T2 = T_translate(0.4, 0.9)

# left: single translation
axes[0].add_patch(Polygon(REF.numpy(), facecolor="lightgray", edgecolor="gray"))
p1 = apply_T(T1, REF).numpy()
axes[0].add_patch(Polygon(p1, facecolor=(0.85, 0.2, 0.2, 0.35), edgecolor="crimson"))
axes[0].set_title("single translation")

# right: composed translations
axes[1].add_patch(Polygon(REF.numpy(), facecolor="lightgray", edgecolor="gray"))
p2 = apply_T(T2 @ T1, REF).numpy()
axes[1].add_patch(Polygon(apply_T(T1, REF).numpy(), facecolor=(0.85, 0.85, 0.2, 0.25), edgecolor="goldenrod"))
axes[1].add_patch(Polygon(p2, facecolor=(0.2, 0.4, 0.85, 0.35), edgecolor="navy"))
axes[1].set_title("composition: $T_2 T_1$")

for ax in axes:
    ax.set_xlim(-0.5, 3.0); ax.set_ylim(-0.5, 3.5); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("Figure 38.3 — translation and composition")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 38.4 — non-uniform (anisotropic) scaling.
fig, ax = plt.subplots(figsize=(4.5, 3.5))
ax.add_patch(Polygon(REF.numpy(), facecolor="lightgray", edgecolor="gray"))
M = T_scale(1.7, 0.55)
ax.add_patch(Polygon(apply_T(M, REF).numpy(), facecolor=(0.85, 0.2, 0.2, 0.35), edgecolor="crimson"))
ax.set_xlim(-0.5, 2.7); ax.set_ylim(-0.5, 2.7); ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("Figure 38.4 — non-uniform scaling ($s_x=1.7,\\, s_y=0.55$)")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 38.5 — rotation by theta counterclockwise around the origin.
fig, ax = plt.subplots(figsize=(4.5, 3.5))
ax.add_patch(Polygon(REF.numpy(), facecolor="lightgray", edgecolor="gray"))
M = T_rotate(np.pi / 4)
ax.add_patch(Polygon(apply_T(M, REF).numpy(), facecolor=(0.85, 0.2, 0.2, 0.35), edgecolor="crimson"))
# rotation arc
arc = np.linspace(0, np.pi / 4, 30)
ax.plot(0.5 * np.cos(arc), 0.5 * np.sin(arc), "k", lw=0.7)
ax.annotate(r"$\theta$", (0.45, 0.05))
ax.set_xlim(-1.5, 2.7); ax.set_ylim(-0.5, 2.7); ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
ax.set_title(r"Figure 38.5 — rotation by $\theta=\pi/4$ around the origin")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 38.6 — three shear examples.
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
shears = [("horizontal", T_shear(0.35, 0.0)), ("vertical", T_shear(0.0, 0.35)), ("both", T_shear(0.25, 0.2))]
for ax, (name, M) in zip(axes, shears):
    ax.add_patch(Polygon(REF.numpy(), facecolor="lightgray", edgecolor="gray"))
    ax.add_patch(Polygon(apply_T(M, REF).numpy(), facecolor=(0.85, 0.2, 0.2, 0.35), edgecolor="crimson"))
    ax.set_xlim(-0.3, 2.4); ax.set_ylim(-0.3, 2.8); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{name} shear")
fig.suptitle("Figure 38.6 — three shear examples")
plt.tight_layout()
plt.show()

### Figure 38.7 — Summary of 2D transformations

The Swiss-army-knife table from the chapter:

| name | matrix | DOF | preserves |
|---|---|---|---|
| translation | $\begin{bmatrix}1 & 0 & t_x\\0 & 1 & t_y\\0 & 0 & 1\end{bmatrix}$ | 2 | distances, angles, orientation |
| Euclidean | $\begin{bmatrix}R & \mathbf{t}\\0 & 1\end{bmatrix}$ ($R\in\mathrm{SO}(2)$) | 3 | distances, angles |
| similarity | $\begin{bmatrix}sR & \mathbf{t}\\0 & 1\end{bmatrix}$ | 4 | angles, ratios of lengths |
| affine | $\begin{bmatrix}a & b & c\\d & e & f\\0 & 0 & 1\end{bmatrix}$ | 6 | parallelism |
| projective | $\begin{bmatrix}a & b & c\\d & e & f\\g & h & i\end{bmatrix}$ | 8 | straight lines, cross-ratio |

Every transformation in this chapter is a special case of projective.

### 38.3.7 — Geometric transformation as a convolution

If you represent an image as a set of triples $\{(\ell_i, x_i, y_i)\}$ (intensity + explicit position), then a rotation is a *1D convolution on the position channels*:

$$\begin{bmatrix}x'\\y'\end{bmatrix} = \underbrace{\begin{bmatrix}\cos\theta & -\sin\theta\\\sin\theta & \cos\theta\end{bmatrix}}_{\text{kernels } w_x, w_y} \begin{bmatrix}x\\y\end{bmatrix}.$$

Intensity passes through untouched; only the coordinate channels are mixed (figure 38.8). This framing is what makes it natural to learn warps with neural networks operating on coordinate channels — the rest of the chapter sets up that bridge.

In [ ]:
# Figure 38.8 — explicit (intensity, x, y) channels; rotation acts only on (x, y).
fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.6))
theta = np.pi / 6
kernel = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
for ax, side in zip(axes, ["input", "output"]):
    ax.set_xticks([]); ax.set_yticks([]); ax.axis("off")
    pts = REF.numpy() if side == "input" else (REF.numpy() @ kernel.T)
    intensity = np.linspace(0.2, 1.0, len(pts))
    ax.scatter(pts[:, 0], pts[:, 1], c=intensity, cmap="viridis", s=90, edgecolor="k")
    ax.set_title(f"{side}: $(\\ell_i, x_i, y_i)$")
    ax.set_aspect("equal")
fig.text(0.5, 0.05,
         r"$[x',y']^\top = [w_x; w_y] [x, y]^\top$ — kernels mix only the coordinate channels",
         ha="center", fontsize=9)
fig.suptitle("Figure 38.8 — geometric transformation reframed as 1D convolution")
plt.tight_layout()
plt.show()

## 38.4 — Lines and points: cross-product duality

In homogeneous coordinates, a 2D line $ax + by + c = 0$ is the vector $\mathbf{l} = (a, b, c)$, and incidence is $\mathbf{l}^\top \mathbf{p} = 0$. Two operations turn out to be the **same** cross product:

$$\mathbf{l} = \mathbf{p}_1 \times \mathbf{p}_2 \qquad \text{(line through two points)}$$

$$\mathbf{p} = \mathbf{l}_1 \times \mathbf{l}_2 \qquad \text{(intersection of two lines)}$$

Parallel lines intersect at a point with $w = 0$ — a *point at infinity*, the homogeneous form of a vanishing point (this is exactly the construction Ch. 47 uses).

In [ ]:
# Figure 38.9 — line through two points, intersection of two lines (same operation).
fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.6))

# Left: line through two points
p1 = torch.tensor([0.5, 0.7])
p2 = torch.tensor([3.0, 2.3])
line = torch.cross(to_homogeneous(p1), to_homogeneous(p2), dim=-1)
xs = np.linspace(0, 3.5, 50)
ys = -(line[0].item() * xs + line[2].item()) / line[1].item()
axes[0].plot(xs, ys, "crimson", lw=1.2)
axes[0].scatter(*p1.numpy(), color="navy", s=70, zorder=5)
axes[0].scatter(*p2.numpy(), color="navy", s=70, zorder=5)
axes[0].text(*p1.numpy(), r" $\mathbf{p}_1$", va="top")
axes[0].text(*p2.numpy(), r" $\mathbf{p}_2$", va="top")
axes[0].set_title(r"$\mathbf{l} = \mathbf{p}_1 \times \mathbf{p}_2$")

# Right: intersection of two lines
l1 = torch.tensor([1.0, -0.5, -0.6])
l2 = torch.tensor([-0.3, 1.0, -1.2])
p_int = from_homogeneous(torch.cross(l1, l2, dim=-1))
for line_vec, color in [(l1, "crimson"), (l2, "seagreen")]:
    a, b, c = line_vec
    if abs(b.item()) > 1e-6:
        ys2 = -(a.item() * xs + c.item()) / b.item()
        axes[1].plot(xs, ys2, color=color, lw=1.2)
axes[1].scatter(*p_int.numpy(), color="navy", s=80, zorder=5)
axes[1].text(*p_int.numpy(), r" $\mathbf{p}=\mathbf{l}_1\times\mathbf{l}_2$", va="top")
axes[1].set_title(r"$\mathbf{p} = \mathbf{l}_1 \times \mathbf{l}_2$")

for ax in axes:
    ax.set_xlim(-0.2, 3.7); ax.set_ylim(-0.2, 3.0); ax.set_aspect("equal")
    ax.grid(True, lw=0.3)
fig.suptitle("Figure 38.9 — points and lines are dual under the cross product")
plt.tight_layout()
plt.show()

## 38.5 — Image warping: forward vs. backward mapping

Applying a transformation $M$ to an image *should* give you a new image. The naive way (forward mapping) walks input pixels, transforms each location, and writes intensity to the nearest target pixel — that leaves **holes**, because the inverse-density of an expansion never covers the target grid cleanly. The correct way (backward mapping) walks *target* pixels and pulls back through $M^{-1}$, so every output pixel gets one well-defined intensity (figures 38.10–38.11).

We'll demonstrate on a small checkerboard so the artifacts are easy to see.

In [ ]:
def make_checkerboard(n=80, squares=8):
    """Small checkerboard image as a tensor in [0, 1]."""
    g = (torch.arange(n)[:, None] // (n // squares)) + (torch.arange(n)[None, :] // (n // squares))
    return (g % 2).float()


def forward_warp(img, M):
    """Forward mapping with nearest-neighbour write. Produces holes."""
    H, W = img.shape
    out = torch.zeros_like(img)
    ys, xs = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
    pts = torch.stack([xs.flatten().float(), ys.flatten().float()], dim=-1)
    new = apply_T(M, pts).round().long()
    new_x, new_y = new[:, 0], new[:, 1]
    mask = (new_x >= 0) & (new_x < W) & (new_y >= 0) & (new_y < H)
    out[new_y[mask], new_x[mask]] = img[ys.flatten()[mask], xs.flatten()[mask]]
    return out


def backward_warp(img, M):
    """Backward mapping with nearest-neighbour sample. Hole-free."""
    H, W = img.shape
    Minv = torch.linalg.inv(M)
    ys, xs = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
    pts = torch.stack([xs.flatten().float(), ys.flatten().float()], dim=-1)
    src = apply_T(Minv, pts).round().long()
    src_x, src_y = src[:, 0], src[:, 1]
    mask = (src_x >= 0) & (src_x < W) & (src_y >= 0) & (src_y < H)
    out = torch.zeros_like(img)
    out.view(-1)[mask] = img[src_y[mask], src_x[mask]]
    return out

In [ ]:
# Figures 38.10 and 38.11 — forward (artefacts) vs backward (clean) mapping.
img = make_checkerboard(96, squares=8)
M = T_rotate(np.pi / 7) @ T_scale(1.3, 1.3) @ T_translate(-48, -48)
M = T_translate(48, 48) @ M  # rotate about centre, then translate back

fwd = forward_warp(img, M)
bwd = backward_warp(img, M)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, im, title in zip(axes, [img, fwd, bwd], ["input", "forward (holes!)", "backward (clean)"]):
    ax.imshow(im.numpy(), cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Figures 38.10 & 38.11 — forward vs backward mapping under rotation + scale")
plt.tight_layout()
plt.show()

## 38.6 — Implicit image representations (SIREN)

Backward mapping at sub-pixel locations needs an interpolation kernel — but only because the image is stored as a discrete grid. **SIREN** [Sitzmann et al., NeurIPS 2020] does away with the grid: it represents an image as a small MLP $f_\theta : (x, y) \to \text{intensity}$, where every layer uses $\sin$ activations. The trained network *is* the image — you can evaluate it at any continuous coordinate.

Geometric warping then becomes a coordinate transform with zero interpolation kernels:
$$\hat\ell(x, y) = f_\theta(M^{-1}(x, y, 1)^\top).$$

We train a tiny SIREN on a small synthetic test image and verify warping by feeding pre-transformed coordinates (figure 38.12). The training is intentionally short — the point is to demonstrate the representation, not push PSNR.

In [ ]:
class SIREN(nn.Module):
    """Tiny implicit image network. Sine-activated MLP per Sitzmann et al., 2020."""

    def __init__(self, hidden=128, layers=4, w0=30.0):
        super().__init__()
        self.w0 = w0
        net = [nn.Linear(2, hidden)]
        for _ in range(layers - 1):
            net.append(nn.Linear(hidden, hidden))
        net.append(nn.Linear(hidden, 1))
        self.net = nn.ModuleList(net)
        self._init_weights()

    def _init_weights(self):
        # First layer scaled by w0; subsequent layers by 1/sqrt(fan_in)/w0 per the paper.
        with torch.no_grad():
            self.net[0].weight.uniform_(-1 / 2, 1 / 2)
            for layer in self.net[1:]:
                bound = (6.0 / layer.in_features) ** 0.5 / self.w0
                layer.weight.uniform_(-bound, bound)

    def forward(self, xy):
        h = torch.sin(self.w0 * self.net[0](xy))
        for layer in self.net[1:-1]:
            h = torch.sin(self.w0 * layer(h))
        return self.net[-1](h)


def make_target_image(n=64):
    """Synthetic test image with smooth + edge structure (stand-in for the tripod scene)."""
    y, x = torch.meshgrid(torch.linspace(-1, 1, n), torch.linspace(-1, 1, n), indexing="ij")
    radial = torch.exp(-3 * (x * x + y * y))
    stripes = 0.5 + 0.5 * torch.sin(8 * x + 4 * y)
    blob = ((x - 0.3) ** 2 + (y + 0.2) ** 2 < 0.15).float() * 0.4
    return (radial * stripes + blob).clamp(0, 1)


def coord_grid(n):
    """Regular [-1, 1]^2 grid of shape (n*n, 2) used as SIREN input."""
    y, x = torch.meshgrid(torch.linspace(-1, 1, n), torch.linspace(-1, 1, n), indexing="ij")
    return torch.stack([x.flatten(), y.flatten()], dim=-1)

In [ ]:
# Train SIREN for ~600 steps on the synthetic image — enough to memorize, not enough to overfit boundaries.
N = 64
target = make_target_image(N).flatten().unsqueeze(-1)
grid = coord_grid(N)

model = SIREN(hidden=128, layers=4, w0=30.0)
optim = torch.optim.Adam(model.parameters(), lr=1e-4)

for step in range(600):
    pred = model(grid)
    loss = ((pred - target) ** 2).mean()
    optim.zero_grad()
    loss.backward()
    optim.step()

print(f"final reconstruction MSE: {loss.item():.4f}")

In [ ]:
# Figure 38.12 — (a) SIREN-encoded image, (b) same network evaluated at rotated+scaled coords.
with torch.no_grad():
    encoded = model(grid).view(N, N).numpy()

    theta = np.pi / 4
    M = T_rotate(theta) @ T_scale(0.5, 0.5)
    Minv = torch.linalg.inv(M)
    warped_coords = apply_T(Minv, grid)
    warped = model(warped_coords).view(N, N).numpy()

fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.6))
axes[0].imshow(encoded, cmap="viridis", interpolation="nearest")
axes[0].set_title("(a) image encoded by SIREN")
axes[1].imshow(warped, cmap="viridis", interpolation="nearest")
axes[1].set_title("(b) rotated 45° + scaled 0.5×")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Figure 38.12 — implicit image, warped via inverse-coordinate evaluation")
plt.tight_layout()
plt.show()

## 38.7 — Concluding remarks

The arc of the chapter is one promotion at a time:

* **Heterogeneous → homogeneous**: every affine and projective transformation becomes a single matrix product.
* **Points and lines → cross-product duality**: incidence, joins, and intersections collapse into one operation. Parallels meet at points with $w=0$ — vanishing points get a coordinate.
* **Discrete grid → implicit function**: with SIREN, the image *is* its parameters. Warping becomes an inverse-coordinate query and interpolation kernels disappear.

Every later geometry chapter — perspective projection (39), stereo (40), homographies (41), single-view metrology (42), structure-from-motion (44), NeRFs (45) — reuses the homogeneous algebra and the point/line duality set up here.